# 02 · Análisis de Reality Fracture para el prerelease

Flujo: carga el set → mira las tablas que te interesen → (después de abrir tus 6 sobres) pega tu pool → genera el reporte HTML.

Todas las funciones vienen de `00_funciones`. Si falta el set en disco, `cargar_set` lo baja solo (2 requests).

In [ ]:
%run ./00_funciones.ipynb

In [ ]:
# ---- Parámetros ----
SET = "reality fracture"   # también sirve el código: "fra"
REFRESCAR = False          # True = vuelve a bajar el set (precios nuevos)

# Después de abrir tus sobres, pega aquí tu pool (una carta por línea, "2 Nombre" o "Nombre").
# Formato de exportación de Arena/MTGO también sirve. Déjalo vacío para el reporte general.
POOL = """
"""

FUENTES = [  # contexto cualitativo que complementa los datos
    ("Guía oficial de prerelease (WotC)", "https://magic.wizards.com/en/news/feature/reality-fracture-prerelease-guide"),
    ("Draftsim: mecánicas y set", "https://draftsim.com/mtg-reality-fracture/"),
    ("Cardsrealm: guía de Limited", "https://mtg.cardsrealm.com/en-us/articles/limited-guide-reality-fracture-draft-prerelease-and-sealed"),
]

In [ ]:
df, meta = cargar_set(SET, forzar=REFRESCAR)
print(f"{meta.get('name')} ({meta.get('code', '').upper()}) · sale {meta.get('released_at')} · {len(df)} cartas")
print(f"Columnas útiles: {list(df.columns)}")
df.head()

## 1. Perfil de colores

Profundidad de cada color en comunes + infrecuentes (lo que más vas a abrir). Mira sobre todo `removal_duro`, `daño/pelea` y `evasivas`.

In [ ]:
perfil_colores(df).style.background_gradient(cmap="Blues", subset=pd.IndexSlice[:, "criaturas":"fixing"])

## 2. Arquetipos

Las doradas infrecuentes son las "cartas señal": si abres una y tienes profundidad en esos colores, es una buena dirección.

In [ ]:
arquetipos(df)

## 3. Removal e interacción

`pct_criaturas_CU_que_mata` cruza el daño del removal con las resistencias del formato: un "2 de daño" que mata 50% vale menos que uno de 3 que mata 70%.

In [ ]:
tabla_interaccion(df, rarezas=("common", "uncommon"))

## 4. Trucos de combate

Todo lo que el rival puede hacer a velocidad de instantáneo en C/U. Filtra por color para memorizar lo que ves en la mesa, ej. `hoja_trucos(df).query("color == 'W'")`.

In [ ]:
hoja_trucos(df)

## 5. Combate: resistencias, tamaños y curva

In [ ]:
display(dureza(df))
display(tamano_por_cmc(df))
curva_criaturas(df)

## 6. Mecánicas

Si en `keywords_top` aparece algo que no está en la primera tabla, agrégalo a `MECANICAS` (en `00_funciones`) y vuelve a correr.

In [ ]:
display(mecanicas_por_color(df))
keywords_top(df)

## 7. Raras y míticas a vigilar

In [ ]:
bombas(df, n=30)

## 8. Qué esperar de 6 sobres

In [ ]:
prob_cartas, prob_removal = probabilidades(df, sobres=6)
display(prob_cartas)
prob_removal

## 9. Tu pool (después de abrir)

In [ ]:
resultado_pool = evaluar_pool(df, POOL) if POOL.strip() else None
if resultado_pool:
    display(resultado_pool["pares"].head(5))
    display(resultado_pool["splash"])
else:
    print("POOL vacío: pega tu pool en la celda de parámetros para evaluarlo.")

## 10. Reporte web

- `reporte_set` → un solo HTML en `reportes/` (ábrelo con doble clic o míralo aquí abajo).
- `exportar_sitio` → carpeta `docs/` lista para GitHub Pages. Súbela al repo y activa *Settings → Pages → main /docs*.

Ambos necesitan la carpeta `web/` del repo en el workspace. Las imágenes se cargan desde Scryfall (hace falta internet).

In [ ]:
ruta = reporte_set(df, meta, pool=resultado_pool, fuentes=FUENTES)
exportar_sitio(df, meta, pool=resultado_pool, fuentes=FUENTES)
mostrar_reporte(ruta)

## Extra: ver cartas del set

In [ ]:
# Cualquier carta del set por nombre (parcial vale). Ejemplo: las 3 raras con mayor puntaje.
mostrar_carta(bombas(df, n=3)["nombre"].tolist(), fuente=SETS / f"{meta['code']}.parquet")